# Week 2 示例：PyTorch Tensor 与训练循环

- 作者：共享仓库示例
- 日期：2026-07-15
- 来源：《实验室新生暑期居家集训学习计划》Week 2，PyTorch 深度学习入门
- 适用周次：Week 2
- 分类：PyTorch
- 关键词：Tensor、autograd、nn.Module、Dataset、DataLoader、训练循环
- 运行环境：Python 3.10+、PyTorch 2.0+；CPU 或 CUDA 均可

## 学习目标

1. 创建 Tensor，并根据设备选择 CPU 或 CUDA。
2. 用 `backward()` 计算自动微分得到的梯度。
3. 用 `Dataset`、`DataLoader` 和 `nn.Module` 组织一个最小训练实验。

> 这是原始计划中 PyTorch 基础代码的最小示例，不包含 MNIST 下载、完整模型保存和所有评估指标。

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('device:', device)

## 1. Tensor 与 autograd

`requires_grad=True` 会让 PyTorch 记录计算图。调用 `backward()` 后，叶子 Tensor 的 `.grad` 保存对应偏导数。训练模型时，优化器会在每个 batch 前清空旧梯度。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x**2 + 2 * x * y + y**3
z.backward()

print('z =', z.item())
print('dz/dx =', x.grad.item())
print('dz/dy =', y.grad.item())
assert x.grad.item() == 10.0
assert y.grad.item() == 31.0

with torch.no_grad():
    print('inference example:', (x + y).item())

## 2. Dataset、DataLoader 和模型

为了让示例不依赖网络和外部数据，这里构造一个二维线性可分数据集。真实作业可以把 `ToyDataset` 换成 MNIST 或自己的文本、图像数据集。

In [ ]:
class ToyDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features.float()
        self.labels = labels.long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


features = torch.randn(240, 2)
labels = (features[:, 0] + 0.8 * features[:, 1] > 0).long()
dataset = ToyDataset(features, labels)
train_set, test_set = random_split(
    dataset, [192, 48], generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=48)

class LinearClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)

    def forward(self, batch_x):
        return self.linear(batch_x)


model = LinearClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
print(model)

## 3. 最小训练循环

每次更新包含五个基本步骤：清空梯度、前向传播、计算损失、反向传播、更新参数。评估阶段使用 `model.eval()` 和 `torch.no_grad()`。

In [ ]:
loss_history = []

for epoch in range(30):
    model.train()
    total_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_y)

    loss_history.append(total_loss / len(train_set))

model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for batch_x, batch_y in test_loader:
        logits = model(batch_x.to(device))
        predictions = logits.argmax(dim=1).cpu()
        correct += (predictions == batch_y).sum().item()
        total += len(batch_y)

test_accuracy = correct / total
print(f'final test accuracy: {test_accuracy:.3f}')
assert test_accuracy > 0.80

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(loss_history)
ax.set(title='Training loss', xlabel='Epoch', ylabel='Cross-entropy')
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 小结

这个例子展示了从数据集、批量加载到模型训练和评估的最短闭环。提交自己的 PyTorch Notebook 时，应进一步说明数据划分、损失函数、优化器、训练曲线和实验结论，并根据导师要求完成 MNIST 等完整实验。